# MeanFlow, from scratch
Juntang Wang, Hacker, CS 2883R, 14 Oct 2026. Paper: Geng et al., *Mean Flows for One-step Generative Modeling*, arXiv 2505.13447.

**Takeaway: learn the average, not the instant.** The network predicts the average velocity u over [r, t], so one jump
z₀ = z₁ − u(z₁, 0, 1) generates a sample. One JVP makes that average trainable. Run all: it retrains while I talk.

In [ ]:
import os, sys, time, math
if "google.colab" in sys.modules and not os.path.exists("meanflow.py"):  # Colab: fetch our code
    os.system("git clone -q --depth 1 --filter=blob:none --sparse https://github.com/qqgjyx/CS-2883R repo")
    os.system("cd repo && git sparse-checkout set 2026-10-14_meanflow")
    os.chdir("repo/2026-10-14_meanflow")
    os.system("pip -q install timm einops")
sys.path.insert(0, os.getcwd())
import torch, torch.nn as nn, torchvision
import matplotlib.pyplot as plt
from meanflow import MeanFlow          # adapted from github.com/haidog-yaqub/MeanFlow
from models.dit import MFDiT

DEV = "cuda" if torch.cuda.is_available() else "cpu"
LIVE = True                            # False: load cached weights/ instead of training
BUDGET = "live"                        # "live": fits a talk on a free T4; "full": the slides (RTX 5080)
STEPS = {"live": dict(toy=3000, img=800), "full": dict(toy=20000, img=3000)}[BUDGET]
FIG = "figs" if BUDGET == "full" else "figs/live"
MF_C, FM_C = "#e8710a", "#8a8a8a"
os.makedirs("weights", exist_ok=True); os.makedirs(FIG, exist_ok=True)
CLOCK = {}
def tic(name): CLOCK[name] = time.time()
def toc(name): CLOCK[name] = time.time() - CLOCK[name]; print(f"{name}: {CLOCK[name]:.1f}s")

## Data
Synthetic: checkerboard and Swiss roll (FFJORD's toy generators). Real-world: MNIST and Fashion-MNIST.

In [ ]:
def checkerboard(n):                   # Grathwohl et al. 2019 (FFJORD toy_data.py)
    x1 = torch.rand(n) * 4 - 2
    x2 = torch.rand(n) - torch.randint(0, 2, (n,)) * 2 + torch.floor(x1) % 2
    return torch.stack([x1, x2], 1) * 2

def swiss_roll(n):                     # sklearn make_swiss_roll(noise=1), coords (x, z), / 5, as in FFJORD
    s = 1.5 * math.pi * (1 + 2 * torch.rand(n))
    return (torch.stack([s * torch.cos(s), s * torch.sin(s)], 1) + torch.randn(n, 2)) / 5

TOYS = {"checkerboard": checkerboard, "swissroll": swiss_roll}

def images(name):                      # whole training set on the GPU, 1x32x32 in [0, 1]
    cls = {"mnist": torchvision.datasets.MNIST, "fashion": torchvision.datasets.FashionMNIST}[name]
    ds = cls(name, train=True, download=True)
    x = nn.functional.interpolate(ds.data[:, None].float() / 255, size=32, mode="bilinear")
    return x.to(DEV), ds.targets.to(DEV)

## Models
Same loss for everything (`MeanFlow.loss`, mode `meanflow`). 2D points are reshaped to 1×1×2 "images";
a small MLP with a u head (average velocity) and a v head (instantaneous velocity) stands in for the DiT.

In [ ]:
class MLP2D(nn.Module):                # u and v heads on a shared trunk; inputs: Fourier features of z, (t, t - r)
    def __init__(self, w=512, depth=5, fourier=64):
        super().__init__()
        self.register_buffer("B", torch.randn(2, fourier, generator=torch.Generator().manual_seed(0)))
        layers, d = [], 2 * fourier + 2
        for _ in range(depth): layers += [nn.Linear(d, w), nn.SiLU()]; d = w
        self.trunk, self.u, self.v = nn.Sequential(*layers), nn.Linear(w, 2), nn.Linear(w, 2)
    def forward(self, x, t, r, y=None, w=None, return_v=True, use_flash_attention=False):
        zb = x.flatten(1) @ self.B
        h = self.trunk(torch.cat([torch.sin(zb), torch.cos(zb), t[:, None], (t - r)[:, None]], 1))
        u = self.u(h).view_as(x)
        return u if not return_v else (u, self.v(h).view_as(x))

def toy_setup(jvp_tangent=(1.0, 0.0)):
    return MLP2D().to(DEV), MeanFlow(channels=1, image_size=1, num_classes=None, mode="meanflow",
                                     normalizer=["mean_std", [0.0], [2.0]], cfg_scale=None,
                                     time_dist=["lognorm", -2.0, 2.0], adaptive_l2_gamma=0.25,  # paper CIFAR-10 recipe
                                     jvp_tangent=jvp_tangent)

def image_setup():
    model = MFDiT(input_size=32, patch_size=4, in_channels=1, dim=192, depth=4, num_heads=4,
                  num_classes=10, max_cfg=7.5).to(DEV)
    return model, MeanFlow(channels=1, image_size=32, num_classes=10, mode="meanflow",
                           normalizer=["minmax", None, None], cfg_scale=[1.0, 7.5],  # CFG folded into the target
                           time_dist=["lognorm", -0.4, 1.0])

In [ ]:
def train(model, mf, batch, steps, bs, amp=False, lr=1e-3):
    opt = torch.optim.AdamW(model.parameters(), lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    model.train()
    for i in range(steps):
        x, c = batch(bs)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp and DEV == "cuda"):
            loss, _ = mf.loss(model, x, c)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    return model.eval()

def fit(name, model, mf, batch, steps, bs, amp=False):
    path = f"weights/{name}_{BUDGET}.pt"
    if LIVE or not os.path.exists(path):
        tic(name); train(model, mf, batch, steps, bs, amp); toc(name)
        torch.save(model.state_dict(), path)
    else:
        model.load_state_dict(torch.load(path, map_location=DEV)); model.eval()
    return model

def toy_batch(gen):
    return lambda bs: (gen(bs).view(bs, 1, 1, 2).to(DEV), None)

In [ ]:
@torch.no_grad()
def mf_steps(model, mf, z, n=1, y=None):        # MeanFlow (u head): z_r = z_t - (t - r) u(z_t, t, r)
    ts = torch.linspace(1, 0, n + 1, device=z.device)
    w = torch.full((len(z),), sum(mf.cfg_w_range) / 2, device=z.device) if mf.use_cfg_w else None  # guidance scale
    for t, r in zip(ts[:-1], ts[1:]):
        T, R = torch.full((len(z),), t.item(), device=z.device), torch.full((len(z),), r.item(), device=z.device)
        z = z - (t - r) * model(z, T, R, y=y, w=w, return_v=False)
    return mf.normer.unnorm(z)

@torch.no_grad()
def fm_steps(model, mf, z, n, y=None, path=False):  # Flow Matching (v head), Euler; optionally the whole path
    ts, P = torch.linspace(1, 0, n + 1, device=z.device), [z]
    w = torch.full((len(z),), sum(mf.cfg_w_range) / 2, device=z.device) if mf.use_cfg_w else None
    for t, r in zip(ts[:-1], ts[1:]):
        T, R = torch.full((len(z),), t.item(), device=z.device), torch.full((len(z),), r.item(), device=z.device)
        z = z - (t - r) * model(z, T, R, y=y, w=w)[1]
        P.append(z)
    return torch.stack([mf.normer.unnorm(p) for p in P]) if path else mf.normer.unnorm(z)

## Synthetic
Train (live), then sample. Same starting noise in every panel. FM = the v head of the same network,
Euler-integrated; MeanFlow = the u head, 1 step.

In [ ]:
TOY_STEPS, TOY_BS = STEPS["toy"], 1024
toys = {}
for name, gen in TOYS.items():
    model, mf = toy_setup()
    toys[name] = fit(f"toy_{name}", model, mf, toy_batch(gen), TOY_STEPS, TOY_BS), mf

In [ ]:
def scatter(ax, x, c):
    ax.scatter(*x.T.cpu(), s=0.3, alpha=0.4, c=c, rasterized=True)
    ax.set(xlim=(-4.5, 4.5), ylim=(-4.5, 4.5), aspect="equal", xticks=[], yticks=[])
    for s in ax.spines.values(): s.set_visible(False)

torch.manual_seed(0)
N = 20000
z1 = torch.randn(N, 1, 1, 2, device=DEV)
COLS = [("data", None), ("fm1", 1), ("fm2", 2), ("fm4", 4), ("fm50", 50), ("mf1", 1)]
LABEL = {"data": "data", "fm1": "FM, 1 step", "fm2": "FM, 2", "fm4": "FM, 4", "fm50": "FM, 50", "mf1": "MeanFlow, 1 step"}
fig, axes = plt.subplots(2, 6, figsize=(12, 4.2))
for row, (name, gen) in zip(axes, TOYS.items()):
    model, mf = toys[name]
    for ax, (col, n) in zip(row, COLS):
        if col == "data": x, c = gen(N), "#333333"
        elif col == "mf1": x, c = mf_steps(model, mf, z1, 1).view(N, 2), MF_C
        else: x, c = fm_steps(model, mf, z1, n).view(N, 2), FM_C
        scatter(ax, x, c)
        f1, a1 = plt.subplots(figsize=(2, 2)); scatter(a1, x, c)
        f1.savefig(f"{FIG}/r_{name}_{col}.png", dpi=200, bbox_inches="tight", pad_inches=0.02); plt.close(f1)
for ax, (col, _) in zip(axes[0], COLS): ax.set_title(LABEL[col])
plt.show()

## Real-world
Train (live), then sample: MeanFlow, 1 step, two samples per class.

In [ ]:
IMG_STEPS, IMG_BS = STEPS["img"], 128
imgs = {}
for name in ("mnist", "fashion"):
    X, Y = images(name)
    def batch(bs, X=X, Y=Y):
        i = torch.randint(0, len(X), (bs,), device=DEV)
        return X[i], Y[i]
    model, mf = image_setup()
    imgs[name] = fit(f"img_{name}", model, mf, batch, IMG_STEPS, IMG_BS, amp=True), mf

In [ ]:
def tile(x):                                    # (k, 1, 32, 32) -> one tall image, k stacked
    return torch.cat(list(x[:, 0].clamp(0, 1).cpu()), 0)

torch.manual_seed(0)
y = torch.arange(10, device=DEV).repeat_interleave(2)
z = torch.randn(20, 1, 32, 32, device=DEV)
fig, axes = plt.subplots(2, 10, figsize=(10, 4))
for row, (name, (model, mf)) in zip(axes, imgs.items()):
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEV == "cuda"):
        x = mf_steps(model, mf, z, 1, y=y).float()
    for k, ax in enumerate(row):
        col = tile(x[2 * k: 2 * k + 2])
        ax.imshow(col, cmap="gray", vmin=0, vmax=1); ax.axis("off")
        plt.imsave(f"{FIG}/c_{name}_{k}.png", col.numpy(), cmap="gray", vmin=0, vmax=1)
plt.show()

## The jump is the average of the path
One network, two heads, same starting noise: the v head integrated over 100 small steps traces the path;
the u head jumps once. If u learned the average of v, the jump lands where the path ends.

In [ ]:
torch.manual_seed(2)
for name, (model, mf) in toys.items():
    z = torch.randn(20000, 1, 1, 2, device=DEV)
    jump, end = mf_steps(model, mf, z, 1).view(-1, 2), fm_steps(model, mf, z, 100).view(-1, 2)
    d = (jump - end).norm(dim=1).median().item()
    d0 = (jump - end[torch.randperm(len(end))]).norm(dim=1).median().item()
    print(f"{name}: median |jump - path end| = {d:.2f}   shuffled pairs = {d0:.2f}")
model, mf = toys["checkerboard"]
z = torch.randn(70, 1, 1, 2, device=DEV)
P = fm_steps(model, mf, z, 100, path=True).view(101, -1, 2).cpu()
J = mf_steps(model, mf, z, 1).view(-1, 2).cpu()
fig, ax = plt.subplots(figsize=(5, 4.4))
ax.plot(P[:, :, 0], P[:, :, 1], c=FM_C, lw=0.8, alpha=0.7)                       # path: v head, 100 steps
for a, b in zip(P[0], J): ax.plot([a[0], b[0]], [a[1], b[1]], c=MF_C, lw=1.0)   # jump: u head, 1 step
ax.scatter(*P[0].T, s=8, c="#bbbbbb", zorder=3)                                 # start: noise
ax.scatter(*J.T, s=10, c=MF_C, zorder=4)
ax.set(aspect="equal", xticks=[], yticks=[]); [s.set_visible(False) for s in ax.spines.values()]
fig.savefig(f"{FIG}/j_checkerboard.png", dpi=250, bbox_inches="tight", pad_inches=0.02); plt.show()

In [ ]:
model, mf = imgs["mnist"]
torch.manual_seed(3)
y = torch.arange(10, device=DEV); z = torch.randn(10, 1, 32, 32, device=DEV)
with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEV == "cuda"):
    top, bot = fm_steps(model, mf, z, 100, y=y).float().clamp(0, 1), mf_steps(model, mf, z, 1, y=y).float().clamp(0, 1)
grid = torch.cat([torch.cat(list(top[:, 0].cpu()), 1), torch.cat(list(bot[:, 0].cpu()), 1)], 0)
plt.imsave(f"{FIG}/j_mnist.png", grid.numpy(), cmap="gray", vmin=0, vmax=1)
print(f"MNIST: mean |1-step - 100-step| per pixel = {(top - bot).abs().mean():.3f}   "
      f"shuffled = {(top - bot[torch.randperm(10)]).abs().mean():.3f}")
plt.imshow(grid, cmap="gray"); plt.axis("off"); plt.show()

## Break the JVP
Same training, wrong tangent for (∂t u, ∂r u). Cached by default: set ABL_LIVE = True to retrain (~100 s each on an RTX 5080).

In [ ]:
ABL_LIVE = False
def overlap(a, b, bins=40, lim=5.0):
    h = [torch.histogramdd(x.cpu().float(), bins=[bins, bins], range=[-lim, lim, -lim, lim]).hist.flatten() / len(x) for x in (a, b)]
    return 1 - 0.5 * (h[0] - h[1]).abs().sum().item()
# multipliers for (dt, dr) in our (z, t, r) order; names use the paper's (z, r, t) order
TANGENTS = {"t1_r0": (1.0, 0.0), "t1_r1": (1.0, 1.0), "t0_r0": (0.0, 0.0), "t0_r1": (0.0, 1.0)}
PAPER = {"t1_r0": "(v, 0, 1) correct", "t1_r1": "(v, 1, 1)", "t0_r0": "(v, 0, 0)", "t0_r1": "(v, 1, 0)"}
torch.manual_seed(0); z = torch.randn(20000, 1, 1, 2, device=DEV); data = checkerboard(20000)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (tag, tg) in zip(axes, TANGENTS.items()):
    model, mf = toy_setup(tg)
    path = f"weights/abl_{tag}_full.pt"
    if tag == "t1_r0": model.load_state_dict(torch.load("weights/toy_checkerboard_full.pt", map_location=DEV))
    elif ABL_LIVE or not os.path.exists(path):
        tic(f"abl_{tag}"); train(model, mf, toy_batch(checkerboard), 20000, TOY_BS); toc(f"abl_{tag}"); torch.save(model.state_dict(), path)
    else: model.load_state_dict(torch.load(path, map_location=DEV))
    x = mf_steps(model.eval(), mf, z, 1).view(-1, 2)
    scatter(ax, x, MF_C); ax.set_title(f"{PAPER[tag]}: overlap {overlap(x, data):.2f}")
    f1, a1 = plt.subplots(figsize=(2, 2)); scatter(a1, x, MF_C)
    f1.savefig(f"{FIG}/a_{tag}.png", dpi=200, bbox_inches="tight", pad_inches=0.02); plt.close(f1)
plt.show()

In [ ]:
print({k: round(v, 1) for k, v in CLOCK.items()}, f"total training {sum(CLOCK.values()):.0f}s")